In [1]:
import numpy as np
import pandas as pd
import json
from random import shuffle, sample
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import spacy
from sklearn.metrics import f1_score

In [2]:
def load_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Preprocess data
def preprocess_data(df):
    df = df[['message', 'sender_annotation']].copy()
    df['sender_annotation'] = df['sender_annotation'].astype(int)
    return df

train_data = preprocess_data(load_data("data/train_sm.jsonl"))
test_data = preprocess_data(load_data("data/test_sm.jsonl"))
validation_data = preprocess_data(load_data("data/validation_sm.jsonl"))

print(train_data.head())

                                             message  sender_annotation
0  Oh, I didn't reply yet, did I?  Yes, I can giv...                  1
1  You’re in the land of the blind...except you h...                  1
2  Well I know this gonna sound a bit greedy righ...                  1
3      I don’t mind helping if it keeps England away                  1
4         I appreciate the help. I’ll consider that.                  1


In [3]:
import joblib

vectorizer = TfidfVectorizer(max_features=500)
X_train_text = vectorizer.fit_transform(train_data['message'])
X_test_text = vectorizer.transform(test_data['message'])
X_validation_text = vectorizer.transform(validation_data['message'])

y_train = train_data['sender_annotation']
y_test = test_data['sender_annotation']
y_validation = validation_data['sender_annotation']


model = LogisticRegression()
model.fit(X_train_text, y_train)


y_test_pred = model.predict(X_test_text)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Classification Report:\n", classification_report(y_test, y_test_pred))


y_validation_pred = model.predict(X_validation_text)
print("Validation Accuracy:", accuracy_score(y_validation, y_validation_pred))
print("Validation Classification Report:\n", classification_report(y_validation, y_validation_pred))

# Save the logistic regression model
joblib.dump(model, "LR_model.pkl")

Test Accuracy: 0.9124407150674936
Test Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       240
           1       0.91      1.00      0.95      2501

    accuracy                           0.91      2741
   macro avg       0.46      0.50      0.48      2741
weighted avg       0.83      0.91      0.87      2741

Validation Accuracy: 0.96045197740113
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        56
           1       0.96      1.00      0.98      1360

    accuracy                           0.96      1416
   macro avg       0.48      0.50      0.49      1416
weighted avg       0.92      0.96      0.94      1416



c:\Users\vimal\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\vimal\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\vimal\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\vimal\anaconda3\Lib\site-packag

['LR_model.pkl']